# QuantiPhy VLM arm - Kaggle / Colab runner

Runs `scripts/run_vlm_job.py`, the *same* script HF Jobs runs. There is no second harness: the job
is environment-driven and checkpoints to the Hub, so a free tier and a paid detached job differ only
in who supplies the GPU.

**Use a free tier for iteration, pay only for the one big pass.** Prompt work means twenty runs over
159 validation rows; that belongs on Kaggle's 30 free GPU-hours a week. A single 3,289-row test pass
belongs on HF Jobs, where detachment and 80 GB actually earn their price.

**On Kaggle, use `Save & Run All` (batch), not an interactive session.** An interactive session dies
with the browser tab. Batch runs detached for up to 12 hours. Enable *Internet* in the sidebar, and
add your `HF_TOKEN` as a Kaggle Secret.

**VRAM.** bf16 needs roughly 2 GB per billion parameters plus frames: an 8B fits Kaggle's T4x2
(2x16 GB) and Colab's A100. A 32B in bf16 needs ~64 GB and fits **neither** - set `VLM_4BIT=1` for
that, and measure what the quantisation costs on validation before trusting a test pass to it.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
!pip install -q "transformers>=4.45" accelerate bitsandbytes opencv-python-headless huggingface_hub pyarrow

## Token

Kaggle: Add-ons > Secrets, name it `HF_TOKEN`. Colab: the key icon in the sidebar.

In [ ]:
import os
try:                                            # Kaggle
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    try:                                        # Colab
        from google.colab import userdata
        os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    except Exception:
        import getpass
        os.environ["HF_TOKEN"] = getpass.getpass("HF token: ")
print("token set:", bool(os.environ.get("HF_TOKEN")))

## Code

In [ ]:
!git clone -q --branch fix/prior-grounding-phrase https://github.com/prarabdhmisra/quantiphy.git
%cd quantiphy

## Run

`RUN_NAME` must change whenever the prompt changes, or the checkpoint resumes and replays stale
replies while appearing to work. Start with `LIMIT=8` to confirm the path before spending an hour.

In [ ]:
%env OUTPUT_REPO=prarabdhmisra/quantiphy-runs
%env SPLIT=validation
%env VLM_MODEL=Qwen/Qwen3-VL-8B-Instruct
%env RUN_NAME=validation-vlm-8b-p1
%env VLM_FRAMES=12
%env LIMIT=8
!python scripts/run_vlm_job.py

## Score it

Raw replies land in `<run>/vlm_raw.jsonl` on the Hub. Re-parsing them is free, so prompt-parsing and
fusion changes never need the GPU again. This scores the run against the organizers' validation truth
using the anchored scorer, which reproduces the published GPT-5.1 macro to 0.4856.

In [ ]:
import json, pandas as pd
from huggingface_hub import hf_hub_download
from quantiphy.scoring import score
from quantiphy.parsing import build_request
from quantiphy.prompting import parse_answer

run = %env RUN_NAME
raw = [json.loads(l) for l in open(hf_hub_download(
    %env OUTPUT_REPO, repo_type="dataset", filename=f"{run}/vlm_raw.jsonl")) if l.strip()]
val = pd.read_csv(hf_hub_download("PaulineLi/QuantiPhy-validation", repo_type="dataset",
                                  filename="validation_dataset.csv"), encoding="utf-8-sig")
val = val[val.ground_truth_posterior.notna()].reset_index(drop=True)

# Re-parse from raw text rather than trusting the run's parsed_value: that is the whole point.
answers = {}
for record in raw:
    row = val.iloc[record["row_index"]]
    answers[record["row_index"]] = parse_answer(
        record["raw_text"], build_request(row).output_unit).value

sub = val.loc[sorted(answers)].copy()
sub["parsed_value"] = [answers[i] for i in sorted(answers)]
print(f"{len(sub)} rows, {sub.parsed_value.notna().sum()} parsed")
print(score(sub))
print("references -- GPT-5.1 0.4856 | zero-vision constant 0.3707 | Qwen3-VL-32B 46.0 on test")